# 06 · Characters API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabrinaaquino/base-batches-workshop/blob/main/notebooks/06-characters.ipynb)

Venice **Characters** are reusable AI personas with name, backstory, system prompt, and (optionally) attached knowledge files. Think Character.ai but private and uncensored.

You can:
- Browse public characters (this notebook)
- Use them in chat completions via `venice_parameters.character_slug`
- Create your own (Pro feature, done in the Venice web UI)

In [ ]:
%pip install --quiet openai requests python-dotenv rich

In [ ]:
import os, sys

# Try Colab secrets first
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("VENICE_API_KEY")
    os.environ["VENICE_API_KEY"] = api_key
except Exception:
    api_key = os.environ.get("VENICE_API_KEY")

if not api_key:
    from getpass import getpass
    api_key = getpass("Paste your Venice API key: ").strip()
    os.environ["VENICE_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI(
    api_key=api_key,
    base_url="https://api.venice.ai/api/v1",
)
print("Connected to Venice ✔")

## 1. List public characters

In [ ]:
import requests

r = requests.get(
    "https://api.venice.ai/api/v1/characters",
    headers={"Authorization": f"Bearer {api_key}"},
    params={"limit": 10},
)
data = r.json()
characters = data.get("data", [])

print(f"Loaded {len(characters)} characters\n")
for c in characters[:10]:
    name = c.get("name", "?")
    slug = c.get("slug", "?")
    desc = (c.get("description") or "")[:80]
    print(f"  {name:<25} {slug:<25} {desc}")

## 2. Filter by category

In [ ]:
r = requests.get(
    "https://api.venice.ai/api/v1/characters",
    headers={"Authorization": f"Bearer {api_key}"},
    params={"categories": "philosophy", "limit": 5},
)
for c in r.json().get("data", []):
    print(f"  {c.get('name')} — {c.get('slug')}")

## 3. Talk to one

Pick a slug from the list above and pass it via `venice_parameters.character_slug`.

In [ ]:
slug = characters[0]["slug"] if characters else "alan-watts"
print(f"Talking to: {slug}\n")

resp = client.chat.completions.create(
    model="venice-uncensored",
    messages=[{"role": "user", "content": "What's the meaning of building onchain?"}],
    extra_body={"venice_parameters": {"character_slug": slug}},
)
print(resp.choices[0].message.content)

## What just happened

You used a fully-formed AI persona without writing a single system prompt yourself. Characters are great for:
- Building consistent agents across many chats
- Letting non-technical teammates create reusable personalities
- Offloading prompt engineering to the Venice character editor

**Next:** [07 · x402 wallet payments](./07-x402-wallet-payments.ipynb)